# Patch Distribitions
In this notebook, we are going to compute the class totals for each dataset and output them in Latex tabels for publication. We are just going to use the UNI embedding for this. The following datasets exist:
  - kather100k
  - spider-colorectal
  - spider-breast
  - spider-skin
  - spider-thorax

In [19]:
from collections import defaultdict

import pandas as pd
import numpy as np

from hproj.data.feature_space import FeatureSpace
from hproj.data.paths import Paths

def compute_class_counts(space: FeatureSpace, labels_map: dict[int, str]) -> pd.DataFrame:
    uniques, counts = np.unique(space.labels, return_counts=True)
    class_counts = {labels_map[str(cls)]: count for cls, count in zip(uniques.tolist(), counts.tolist())}
    df = pd.DataFrame.from_dict(class_counts, orient='index', columns=['count'])
    df.reset_index(inplace=True)
    df.columns = ['label', 'count']
    return df

encoder = 'uni'
datasets = [
    'kather100k',
    'spider-colorectal',
    'spider-breast',
    'spider-skin',
    'spider-thorax'
]

paths = Paths.from_env()

class_counts = {}

for dataset in datasets:
    embeddings_path = paths.embedding(dataset, encoder)
    labels_map = embeddings_path.load_label_map()

    train, test = embeddings_path.load_splits()
    train_class_counts = compute_class_counts(train, labels_map)
    test_class_counts = compute_class_counts(test, labels_map)
    class_counts[dataset] = {
        'train': train_class_counts,
        'test': test_class_counts
    }

In [20]:
from pylatex import Document, LongTable, MultiColumn
from pylatex.utils import bold

for dataset in datasets:
    train_df = class_counts[dataset]['train']
    test_df = class_counts[dataset]['test']
    
    # Merge train and test counts
    merged_df = train_df.merge(test_df, on='label', suffixes=('_train', '_test'))
    merged_df['total'] = merged_df['count_train'] + merged_df['count_test']
    
    doc = Document()
    
    with doc.create(LongTable('l r r r')) as table:
        # Header
        table.add_hline()
        table.add_row([bold('Class'), bold('Train'), bold('Test'), bold('Total')])
        table.add_hline()
        table.end_table_header()
        
        # Rows
        for _, row in merged_df.iterrows():
            table.add_row([row['label'], row['count_train'], row['count_test'], row['total']])
        
        # Footer
        table.add_hline()
        table.add_row([
            bold('Total'),
            bold(merged_df['count_train'].sum()),
            bold(merged_df['count_test'].sum()),
            bold(merged_df['total'].sum())
        ])
        table.add_hline()
    
    filename = f'class_counts_{dataset}'
    doc.generate_tex(filename)
    print(f'Generated {filename}.tex')

Generated class_counts_kather100k.tex
Generated class_counts_spider-colorectal.tex
Generated class_counts_spider-breast.tex
Generated class_counts_spider-skin.tex
Generated class_counts_spider-thorax.tex
